In [1]:
# coding: utf-8
import sys
sys.path.append('..')
from common.np import *  # import numpy as np
from common.layers import Embedding, SigmoidWithLoss
import collections


class EmbeddingDot:
    def __init__(self, W):
        self.embed = Embedding(W)  # Embeddingクラスをself.embedとして呼び出せるようにする
        self.params = self.embed.params  # self.embed.paramsにある重みを、self.paramsとして呼び出す
        self.grads = self.embed.grads  # self.embed.gradsにある勾配を、self.gradsとして呼び出す
        self.cache = None

    def forward(self, h, idx):
        target_W = self.embed.forward(idx)  # idx番目の重みを抜き出して、target_Wとして呼び出す
        out = np.sum(target_W * h, axis=1)  # target_Wと周りの単語を混ぜたベクトル（h）を掛けてスコアを出す

        self.cache = (h, target_W)  # 後のbackwardで使うため、スコアの材料をself.cacheで呼び出せるようにしておく
        return out

    def backward(self, dout):
        h, target_W = self.cache  # スコアの材料を取り出す
        dout = dout.reshape(dout.shape[0], 1)  # doutを、ベクトルとの掛け算ができる形に変形する

        dtarget_W = dout * h  # doutにhをかけて勾配を計算する
        self.embed.backward(dtarget_W)  # dtarget_Wの勾配をEmbeddingレイヤに渡す
        dh = dout * target_W # ズレに単語の重みを掛ける
        return dh


class UnigramSampler:
    def __init__(self, corpus, power, sample_size):
        self.sample_size = sample_size
        self.vocab_size = None
        self.word_p = None

        counts = collections.Counter()
        for word_id in corpus:
            counts[word_id] += 1

        vocab_size = len(counts)
        self.vocab_size = vocab_size

        self.word_p = np.zeros(vocab_size)
        for i in range(vocab_size):
            self.word_p[i] = counts[i]

        self.word_p = np.power(self.word_p, power)
        self.word_p /= np.sum(self.word_p)

    def get_negative_sample(self, target):
        batch_size = target.shape[0]

        if not GPU:
            negative_sample = np.zeros((batch_size, self.sample_size), dtype=np.int32)

            for i in range(batch_size):
                p = self.word_p.copy()
                target_idx = target[i]
                p[target_idx] = 0
                p /= p.sum()
                negative_sample[i, :] = np.random.choice(self.vocab_size, size=self.sample_size, replace=False, p=p)
        else:
            # GPU(cupy）で計算するときは、速度を優先
            # 負例にターゲットが含まれるケースがある
            negative_sample = np.random.choice(self.vocab_size, size=(batch_size, self.sample_size),
                                               replace=True, p=self.word_p)

        return negative_sample


class NegativeSamplingLoss:
    def __init__(self, W, corpus, power=0.75, sample_size=5):
        self.sample_size = sample_size
        self.sampler = UnigramSampler(corpus, power, sample_size)  # corpusから5つハズレを選ぶ
        self.loss_layers = [SigmoidWithLoss() for _ in range(sample_size + 1)] # 判定機を6つ分用意
        self.embed_dot_layers = [EmbeddingDot(W) for _ in range(sample_size + 1)] # 計算機を6つ分用意

        self.params, self.grads = [], []    # 重みと勾配を1つのリストにまとめる
        for layer in self.embed_dot_layers:
            self.params += layer.params
            self.grads += layer.grads

    def forward(self, h, target):
        batch_size = target.shape[0]
        negative_sample = self.sampler.get_negative_sample(target)  # ハズレ単語を5つ選ぶ指示を出す

        # 正例のフォワード
        score = self.embed_dot_layers[0].forward(h, target)  # 0番目にターゲットを配置
        correct_label = np.ones(batch_size, dtype=np.int32)  # batch_size分正解を1に設定する
        loss = self.loss_layers[0].forward(score, correct_label)  # 0番目がスコア1に近いかを判定、最初のズレを出す

        # 負例のフォワード
        negative_label = np.zeros(batch_size, dtype=np.int32) # batch_size分のハズレ5つを0に設定する
        for i in range(self.sample_size):  # sample_sizeのハズレ5つを1つずつ取り出す
            negative_target = negative_sample[:, i]  # 取り出した値をnegative_targetに格納する
            score = self.embed_dot_layers[1 + i].forward(h, negative_target)  # hとtargetを見て予測値を出力
            loss += self.loss_layers[1 + i].forward(score, negative_label)  # 正解とどれだけズレていたのか判定

        return loss

    def backward(self, dout=1):
        dh = 0
        for l0, l1 in zip(self.loss_layers, self.embed_dot_layers):
            dscore = l0.backward(dout)
            dh += l1.backward(dscore)

        return dh

In [2]:
class Embedding:
    def __init__(self, W):
        self.params = [W]
        self.grads = [np.zeros_like(W)]
        self.idx = None

    def forward(self, idx):
        W, = self.params   # W = self.params[0]と同じ意味
        self.idx = idx
        out = W[idx]   # Wからidx番目の重みを取り出す
        return out

    def backward(self, dout):
        dW, = self.grads
        dW[...] = 0
        if GPU:
            np.scatter_add(dW, self.idx, dout)
        else:
            np.add.at(dW, self.idx, dout)
        return None

In [ ]:
# coding: utf-8
import sys
sys.path.append('..')  # 親ディレクトリのファイルをインポートするための設定
from common.layers import *
from ch04.negative_sampling_layer import NegativeSamplingLoss


class SkipGram:
    # 語彙数、隠れ層のニューロンの数、周辺単語の範囲、学習データ
    def __init__(self, vocab_size, hidden_size, window_size, corpus):
        V, H = vocab_size, hidden_size  # V:語彙数　H:隠れ層のニューロンの数
        rn = np.random.randn

        # 重みの初期化
        W_in = 0.01 * rn(V, H).astype('f')  # 単語をベクトルに変換する
        W_out = 0.01 * rn(V, H).astype('f')  # ベクトルを単語に戻す

        # レイヤの生成
        self.in_layer = Embedding(W_in)  # Embeddingレイヤをself.in_layerとして呼び出す
        self.loss_layers = []  # self.loss_layerの箱を用意(1つの正解と5つのハズレを入れる箱)
        for i in range(2 * window_size):  # ターゲットの周辺単語セットを順番に取り出す
            layer = NegativeSamplingLoss(W_out, corpus, power=0.75, sample_size=5) # ターゲットの予測値を出力
            self.loss_layers.append(layer)  # loss_layerに予測値をまとめる

        # すべての重みと勾配をリストにまとめる
        # in_layerとloss_layerをリストでくっつける
        # （in_layerは単体のレイヤ、loss_layerはリストだから足してまとめる）
        layers = [self.in_layer] + self.loss_layers
        self.params, self.grads = [], []  # 重み用、勾配用の箱を用意
        for layer in layers:  # layersからレイヤーを1つずつ取り出す
            self.params += layer.params  # 重みはparamsへ、
            self.grads += layer.grads    # 勾配はgradsへ格納

        # メンバ変数に単語の分散表現を設定
        self.word_vecs = W_in

    def forward(self, contexts, target):
        h = self.in_layer.forward(target)  # Embeddingで意味ベクトル「h」を生成

        loss = 0  # スコアの初期化
        # loss_layer→NegativeSamplingLossレイヤ（正解判定用1つ、ハズレ判定用5つが入ってる）
        for i, layer in enumerate(self.loss_layers):
            # hとcontexts(ターゲット前後の値)を1つずつ照らし合わせて、どれだけズレてるのかを
            # NegativeSamplingLossレイヤで採点する
            # lossに足す
            # loss=今回のcontexts全体のズレ
            loss += layer.forward(h, contexts[:, i])  
        return loss

    def backward(self, dout=1):
        dh = 0  # 勾配用の箱を用意
        for i, layer in enumerate(self.loss_layers): 
            # レイヤを順番に回り、勾配を計算する
            # 計算した勾配はdhに足し合わせる
            # 一番右のレイヤだけが全体のズレを知ってる、
            # 左には右のレイヤの勾配（dh）が伝わっていく
            dh += layer.backward(dout)
        # 一番左の隠れ層（Embeddingレイヤ）は、右から伝わってきた最終的な勾配を受け取る
        # 受け取った勾配を、自分のgradsにメモする
        # この時点で重みはまだ書き換えない
        self.in_layer.backward(dh)
        return None

In [ ]:
# coding: utf-8
import sys
sys.path.append('..')
import numpy as np
from common.layers import MatMul, SoftmaxWithLoss


class SimpleCBOW:
    def __init__(self, vocab_size, hidden_size):
        # V:語彙数　H:ベクトルの次元数
        V, H = vocab_size, hidden_size

        # 重みの初期化
        W_in = 0.01 * np.random.randn(V, H).astype('f')   # 単語（one-hot）をベクトルに変える（第4章のEmbedding層）
        W_out = 0.01 * np.random.randn(H, V).astype('f')  # ベクトルを単語に戻す

        # レイヤの生成
        self.in_layer0 = MatMul(W_in)              # 第4章ではMatMulレイヤからEmbeddingレイヤに差し変わる
        self.in_layer1 = MatMul(W_in)             # 入力層が2つあるのは、ターゲットの前後の単語を同時に処理したいから
        self.out_layer = MatMul(W_out)
        self.loss_layer = SoftmaxWithLoss()  # 第4章ではNegative Samplingに変わる

        # すべての重みと勾配をリストにまとめる
        layers = [self.in_layer0, self.in_layer1, self.out_layer]
        self.params, self.grads = [], []  # 重み（params）とズレ（grads）それぞれリストを作成
        for layer in layers:  # layersにまとめたレイヤを1つ1つ取り出して、
            self.params += layer.params  # 重みはparamsに、
            self.grads += layer.grads    # ズレはgradsにそれぞれ代入

        # メンバ変数に単語の分散表現を設定
        self.word_vecs = W_in  # W_inを入れるのは、単語ベクトルを保持し続けてくれるから、使いまわしやすい

    def forward(self, contexts, target):
        h0 = self.in_layer0.forward(contexts[:, 0])  # 前の単語を行列のかけ算（MatMul）に通す
        h1 = self.in_layer1.forward(contexts[:, 1])  # 後ろの単語を行列かけ算に通す
        h = (h0 + h1) * 0.5  # h0とh1の情報を混ぜて2で割る（推測するための共通のヒントを作るため）
        score = self.out_layer.forward(h)  # 合体したヒントhに、もう1枚の重みW_outをかける
        loss = self.loss_layer.forward(score, target) # スコアと正解を比べて、どれくらい外れたかの数値を出す
        return loss

    def backward(self, dout=1):
        ds = self.loss_layer.backward(dout)  # dsに順伝播で出たズレを渡す
        da = self.out_layer.backward(ds)
        da *= 0.5
        self.in_layer1.backward(da)
        self.in_layer0.backward(da)
        return None

In [4]:
# word2vecの高速化
# word2vecの改良１
# Embeddingレイヤ

In [ ]:
# Embeddingレイヤの実装
class Embedding:
    def __init__(self, W):
        # paramsとgradsをリストにしておくのは、
        # Optimizerで一気に更新できるようにするため
        # Optimizer=gradsメモ（勾配）を一気に更新する
        self.params = [W]  
        self.grads = [np.zeros_like(W)]
        # 後でインデックスを受け取るための箱を準備しておく
        self.idx = None

    def forward(self, idx):
        W, = self.params  # 重みを取り出す
        self.idx = idx  # idxをself.idxとして受け取れるようにする
        out = W[idx]  # 重み行列から意味ベクトル（h）を抜き出す
        return out